# Descriptive Analysis Tables

This notebook creates aggregate descriptive analysis outputs after the SQL cohort and validation workflow has been reviewed. Outputs are generated from Synthea synthetic data and do not contain patient-level records.

## Load Final Analysis Dataset

The notebook rebuilds the SQL views locally from the committed SQL scripts and loads the final analysis dataset into memory. Patient-level data are used only during local computation and are not exported.

In [ ]:

from pathlib import Path
import sys
import duckdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analysis"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SQL_SCRIPTS = [
    "sql/01_profile_source_tables.sql",
    "sql/02_define_eligible_inpatient_encounters.sql",
    "sql/03_define_index_encounter.sql",
    "sql/04_define_postdischarge_utilization.sql",
    "sql/05_define_30_day_readmission.sql",
    "sql/06_create_final_analysis_dataset.sql",
]

def load_final_analysis_dataset():
    db_path = "/tmp/ehr_readmission_analysis_notebook.duckdb"
    con = duckdb.connect(db_path)
    for script in SQL_SCRIPTS:
        con.execute((PROJECT_ROOT / script).read_text())
    df = con.execute("SELECT * FROM final_analysis_dataset").fetchdf()
    con.close()
    return df

df = load_final_analysis_dataset()
print(f"Loaded final analysis dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")


## Table 1 Baseline Characteristics

Table 1 summarizes demographic, utilization, and comorbidity characteristics overall and by 30-day readmission status.

In [ ]:

from scipy import stats

outcome = "readmitted_30d"
readmitted = df[df[outcome] == 1]
not_readmitted = df[df[outcome] == 0]

continuous_vars = [
    ("age_at_index", "Age at index, years"),
    ("length_of_stay_days", "Length of stay, days"),
    ("prior_encounters_12mo", "Prior encounters, 12 months"),
    ("prior_ed_visits_12mo", "Prior ED visits, 12 months"),
    ("chronic_condition_count", "Chronic condition count"),
]

categorical_vars = [
    ("sex", "Sex"),
    ("race", "Race"),
    ("ethnicity", "Ethnicity"),
    ("age_group_65plus", "Age group 65+"),
    ("diabetes_flag", "Diabetes flag"),
    ("hypertension_flag", "Hypertension flag"),
    ("ckd_flag", "CKD flag"),
    ("copd_flag", "COPD flag"),
    ("outpatient_followup_30d", "Outpatient follow-up within 30 days"),
    ("ed_revisit_30d", "ED revisit within 30 days"),
]

def mean_sd(series):
    clean = pd.to_numeric(series, errors="coerce").dropna()
    if clean.empty:
        return "Missing"
    return f"{clean.mean():.2f} ({clean.std(ddof=1):.2f})"

def n_pct(count, denom):
    return f"{int(count)} ({100 * count / denom:.1f}%)" if denom else "0 (0.0%)"

rows = []
for col, label in continuous_vars:
    a = pd.to_numeric(not_readmitted[col], errors="coerce").dropna()
    b = pd.to_numeric(readmitted[col], errors="coerce").dropna()
    p_value = np.nan
    test = "Mann-Whitney U"
    if len(a) > 0 and len(b) > 0:
        try:
            p_value = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        except ValueError:
            p_value = np.nan
    rows.append({
        "variable": label,
        "level": "Mean (SD)",
        "overall": mean_sd(df[col]),
        "not_readmitted": mean_sd(not_readmitted[col]),
        "readmitted": mean_sd(readmitted[col]),
        "p_value": round(p_value, 4) if pd.notna(p_value) else "",
        "test": test,
    })

for col, label in categorical_vars:
    levels = sorted(df[col].dropna().unique().tolist())
    p_value = ""
    test = "Chi-square"
    contingency = pd.crosstab(df[col], df[outcome])
    if contingency.shape[0] > 1 and contingency.shape[1] == 2:
        try:
            p_value = round(stats.chi2_contingency(contingency)[1], 4)
        except ValueError:
            p_value = ""
    for i, level in enumerate(levels):
        overall_count = (df[col] == level).sum()
        not_count = (not_readmitted[col] == level).sum()
        readmit_count = (readmitted[col] == level).sum()
        rows.append({
            "variable": label if i == 0 else "",
            "level": str(level),
            "overall": n_pct(overall_count, len(df)),
            "not_readmitted": n_pct(not_count, len(not_readmitted)),
            "readmitted": n_pct(readmit_count, len(readmitted)),
            "p_value": p_value if i == 0 else "",
            "test": test if i == 0 else "",
        })

table1 = pd.DataFrame(rows)
table1.to_csv(OUTPUT_DIR / "table1_baseline_characteristics.csv", index=False)
table1.head(20)


## Readmission Summary

In [ ]:

readmission_summary = pd.DataFrame([{
    "cohort_patients": len(df),
    "readmitted_30d_count": int(df["readmitted_30d"].sum()),
    "readmission_rate_30d_percent": round(100 * df["readmitted_30d"].mean(), 1),
    "mean_days_to_readmission": round(df["days_to_readmission"].mean(), 2),
    "median_days_to_readmission": round(df["days_to_readmission"].median(), 2),
    "min_days_to_readmission": round(df["days_to_readmission"].min(), 2),
    "max_days_to_readmission": round(df["days_to_readmission"].max(), 2),
}])
readmission_summary.to_csv(OUTPUT_DIR / "readmission_summary.csv", index=False)
readmission_summary


## Outpatient Follow-Up Summary

In [ ]:

followup_rows = []
for label, days, col in [
    ("0-7 days", 7, "outpatient_followup_7d"),
    ("0-14 days", 14, "outpatient_followup_14d"),
    ("0-30 days", 30, "outpatient_followup_30d"),
]:
    followup_rows.append({
        "followup_window": label,
        "followup_window_days": days,
        "followup_count": int(df[col].sum()),
        "followup_percent": round(100 * df[col].mean(), 1),
        "mean_days_to_first_followup": round(df.loc[df[col] == 1, "days_to_first_outpatient_followup"].mean(), 2),
        "median_days_to_first_followup": round(df.loc[df[col] == 1, "days_to_first_outpatient_followup"].median(), 2),
    })
followup_summary = pd.DataFrame(followup_rows)
followup_summary.to_csv(OUTPUT_DIR / "outpatient_followup_summary.csv", index=False)
followup_summary


## ED Revisit Summary

In [ ]:

ed_revisit_summary = pd.DataFrame([{
    "cohort_patients": len(df),
    "ed_revisit_30d_count": int(df["ed_revisit_30d"].sum()),
    "ed_revisit_30d_percent": round(100 * df["ed_revisit_30d"].mean(), 1),
    "any_postdischarge_encounter_30d_count": int((df["total_postdischarge_encounters_30d"] > 0).sum()),
    "any_postdischarge_encounter_30d_percent": round(100 * (df["total_postdischarge_encounters_30d"] > 0).mean(), 1),
    "mean_total_postdischarge_encounters_30d": round(df["total_postdischarge_encounters_30d"].mean(), 2),
}])
ed_revisit_summary.to_csv(OUTPUT_DIR / "ed_revisit_summary.csv", index=False)
ed_revisit_summary


## Output Files

In [ ]:

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(path.relative_to(PROJECT_ROOT))
